# Station Stacking v9 - KATL

Experimental notebook for `KATL`.

This version keeps the v8 live-safe GFS/HRRR/NBM contract, adds source-owned 10-year calendar-day max-temperature climatology features, and trains base learners on remaining warmup from the observed high-so-far. Artifacts are written to `data/calibration/station_stacking_v9`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v9_remaining_warmup"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V9_DROPPED_FEATURE_COLUMNS,
    V9_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V9 Contract

`feature_version="v9"` keeps the v8 remaining-warmup feature set, adds leakage-safe 10-year calendar-day climatology features, and uses `target_mode="remaining_warmup"` so base learners fit `actual_high_f - observed_high_temp_through_as_of_f` before converting back to `predicted_high_f`.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V9_FEATURE_COLUMNS, sorted(V9_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1987,2021-01-01,2026-06-10
1,KATL,hrrr,1987,2021-01-01,2026-06-10
2,KATL,nbm,1986,2021-01-01,2026-06-10


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v9",
    target_mode="remaining_warmup",
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v9/KATL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-16 07:22:10,960] Using an existing study with name 'KATL_v9_remaining_warmup_base_xgboost_mae_f' instead of creating a new one.
[I 2026-06-16 07:22:11,064] Using an existing study with name 'KATL_v9_remaining_warmup_base_lightgbm_mae_f' instead of creating a new one.
[I 2026-06-16 07:22:11,155] Using an existing study with name 'KATL_v9_remaining_warmup_base_catboost_mae_f' instead of creating a new one.
d:\dev\weather-research\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
d:\dev\weather-research\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
d:\dev\weather-research\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warni

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,728,1.838715,2.553950
1,validation_2024_2025,lightgbm,728,1.755051,2.469478
2,validation_2024_2025,catboost,728,1.623338,2.300644
3,validation_2024_2025,hrrr_raw,728,3.307569,4.576736
4,validation_2024_2025,gfs_raw,728,3.209027,4.407911
5,test_2026,xgboost,137,1.861826,2.617573
6,test_2026,lightgbm,137,1.663339,2.327628
7,test_2026,catboost,137,1.600847,2.257958
8,test_2026,ridge_stack,137,1.590929,2.252103
9,test_2026,hrrr_raw,137,3.620798,5.304939


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    source_pipeline="notebooks/station_stacking_v9",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v9/model_weights/KATL_station_high_regressor_v9_remaining_warmup.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v9/model_weights/KATL_station_high_regressor_v9_remaining_warmup.json'))

## V9 Feature Coverage


In [9]:
v9_feature_coverage = (
    result.features[V9_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v9_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_observed_precip_recent_mm_est,100.000000
6,v4_forecast_wet_observed_dry,100.000000
7,v4_forecast_observed_precip_match,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V9_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
163,v2_recent_heat_anomaly_f,numeric
164,v2_recent_heat_momentum_f,numeric
165,v2_morning_warmup_to_consensus_f,numeric
166,v2_consensus_minus_7d_actual_f,numeric
167,v2_spread_per_warmup_f,numeric
168,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V9_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,99.898167
1,observed_temp_change_last_3h_f,99.898167
2,observed_morning_warmup_rate_f_per_hour,99.898167
3,observed_high_so_far_change_since_9am_f,99.898167


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
3,oof_2026,lightgbm,137,80,58.394161
5,oof_2026,ridge_stack,137,78,56.934307
0,oof_2026,catboost,137,77,56.204380
6,oof_2026,xgboost,137,72,52.554745
4,oof_2026,nbm_raw,137,54,39.416058
1,oof_2026,gfs_raw,137,40,29.197080
2,oof_2026,hrrr_raw,137,40,29.197080
7,validation_2024_2025,catboost,728,437,60.027473
10,validation_2024_2025,lightgbm,728,409,56.181319
12,validation_2024_2025,xgboost,728,384,52.747253


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,lightgbm,125,1.547989,2.093915,v5
1,test_2026,ridge_stack,125,1.566708,2.118374,v5
2,test_2026,xgboost,125,1.569277,2.117086,v5
3,test_2026,lightgbm,125,1.575729,2.153196,v6
4,test_2026,xgboost,125,1.588664,2.125651,v6
...,...,...,...,...,...,...
94,validation_2024_2025,gfs_raw,541,3.557627,5.060559,v1
95,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v2
96,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v3
97,validation_2024_2025,hrrr_raw,664,3.630969,5.083107,v4


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,137,1.861826,2.617573,37.226277
1,lightgbm,137,1.663339,2.327628,43.065693
2,catboost,137,1.600847,2.257958,40.875912
3,ridge_stack,137,1.590929,2.252103,39.416058
4,hrrr_raw,137,3.620798,5.304939,21.89781
5,gfs_raw,137,3.312961,4.558053,22.627737
